# Gradient similarity

Pairwise similarity of FGSM input gradients between models, with the attacksv2 configuration system (grain = family/architecture/variant, `mean`/`most_similar`/`least_similar` aggregation, per-view blacklist, asymmetric row/column grains).

**Compute is decoupled from figures**: run `scripts/compute_gradient_similarity_cache.py` on the cluster (Discord-notified, resumable), sync `logs/GRADIENTS/` back, then run this notebook. Pair metrics include **all** models (blacklisted ones too); exclude per view via `model_blacklist=`/`blacklist_contains=` in the config.

Metrics (higher = more similar): `Sign Agreement` (most FGSM-relevant, random ~0.5), `Gradient Cosine`, `Saliency TopK Jaccard`, `Saliency Pearson`.

In [ ]:
import sys
from pathlib import Path

# Add src/ to sys.path before importing analytics (required for Jupyter kernels).
_project_root = Path.cwd().resolve()
while _project_root != _project_root.parent and not (_project_root / "src" / "analytics").is_dir():
    _project_root = _project_root.parent
if not (_project_root / "src" / "analytics").is_dir():
    raise FileNotFoundError(
        "Could not find the project root (expected src/analytics/). "
        "Set the notebook working directory to the QB-PIER repo root."
    )
_src = _project_root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from analytics.notebook_setup import notebook_paths

project_root, TRAINING_ROOT, ATTACK_EXPERIMENT_DIR, OUT, _src = notebook_paths("gradient_similarity", start=_project_root)

CACHE_DIR = project_root / "logs" / "GRADIENTS"  # synced from the cluster
METRICS = None        # None = all four similarity metrics; or e.g. ["Sign Agreement"]
RANKING_TOP_K = 3     # rows in the most/least-similar ranking tables
REBUILD_PAIRS = False # True = recompute the pair CSV from cached .pt gradients

## Crawl: cache state

Inspect what the cluster produced before loading anything: run metadata, cached models, and trained models still missing from the cache.

In [ ]:
import json

from analytics.attacks.gradient_similarity import GRADIENTS_SUBDIR, safe_model_name

meta_path = CACHE_DIR / "meta.json"
if meta_path.exists():
    print(json.dumps(json.loads(meta_path.read_text(encoding="utf-8")), indent=2))
else:
    print(f"No meta.json under {CACHE_DIR} - run scripts/compute_gradient_similarity_cache.py")

cached = sorted(p.stem for p in (CACHE_DIR / GRADIENTS_SUBDIR).glob("*.pt")) if (CACHE_DIR / GRADIENTS_SUBDIR).is_dir() else []
trained = sorted(
    p.parent.relative_to(TRAINING_ROOT).as_posix()
    for p in TRAINING_ROOT.rglob(".model")
)
missing = [name for name in trained if safe_model_name(name) not in set(cached)]
print(f"\nCached gradients: {len(cached)} models | trained models: {len(trained)} | missing from cache: {len(missing)}")
for name in missing:
    print("  missing:", name)

## Load: pair metrics

In [ ]:
from analytics.attacks.gradient_similarity import load_gradient_pair_metrics

pair_df = load_gradient_pair_metrics(CACHE_DIR, rebuild=REBUILD_PAIRS)
print(f"{len(pair_df)} pair rows, {pair_df['Source Model'].nunique()} models, n_samples={pair_df['n_samples'].iloc[0]}")
pair_df.head()

## Generate: standard report

Per metric: matrix CSVs for every template (`variant_full` stays CSV-only - the variant heatmap is unreadable), heatmap figures for the family/architecture grains, and top-k most/least similar ranking tables at architecture grain. Outputs under `outputs/gradient_similarity/<metric>/`.

In [ ]:
from analytics.attacks.gradient_similarity import (
    GRADSIM_METRICS,
    GRADSIM_TEMPLATES,
    run_gradient_similarity_report,
)

outputs = run_gradient_similarity_report(
    pair_df,
    out_root=OUT,
    metrics=METRICS or GRADSIM_METRICS,
    ranking_top_k=RANKING_TOP_K,
)
print(f"{len(outputs)} artifacts written under {OUT}")

## Example: quantum variants (rows) vs all architectures (columns)

Asymmetric grains: `source_grain="variant"` + `source_family_filter="Quantum"` selects quantum variants as rows, `target_grain="architecture"` puts every architecture in the zoo as columns. Swap any grain/subset with `source_family_filter`, `selected_rows`, `selected_columns`, `source_contains`, or `model_blacklist`/`blacklist_contains` overrides.

In [ ]:
from analytics.attacks.gradient_similarity import get_gradsim_template
from analytics.attacks.transfer_role_tools import build_attack_matrix, plot_attack_matrix

cfg = get_gradsim_template("quantum_variants_vs_architectures", metric="Sign Agreement")
matrix = build_attack_matrix(pair_df, cfg)
fig, ax = plot_attack_matrix(matrix, title="Sign Agreement - quantum variants vs architectures")
matrix